# Quantum Computing Labs - Combined
## IDTB090046 - Em Sereyvathna

This notebook combines all five quantum computing laboratory exercises:
1. Lab 1: QFT (Quantum Fourier Transform)
2. Lab 2: BV (Bernstein-Vazirani Algorithm with secret number input)
3. Lab 3: QPE (Quantum Phase Estimation)
4. Lab 4: Grover's Search on 3 qubits
5. Lab 5: Shor's Factoring Algorithm

## Setup and Imports

First, let's import all necessary libraries for quantum computing simulations.

In [ ]:
# Import necessary libraries
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from qiskit.quantum_info import Statevector
import numpy as np
import matplotlib.pyplot as plt
from math import gcd, pi
from fractions import Fraction

# Set up the simulator
simulator = Aer.get_backend('qasm_simulator')
statevector_simulator = Aer.get_backend('statevector_simulator')

print("All libraries imported successfully!")

---
# Lab 1: Quantum Fourier Transform (QFT)

The Quantum Fourier Transform is the quantum analogue of the discrete Fourier transform. It's a key component in many quantum algorithms.

## QFT Implementation

We'll implement the QFT for n qubits with proper phase rotations.

In [ ]:
def qft_rotations(circuit, n):
    """Performs qft on the first n qubits in circuit (without swaps)"""
    if n == 0:
        return circuit
    n -= 1
    circuit.h(n)
    for qubit in range(n):
        circuit.cp(pi/2**(n-qubit), qubit, n)
    qft_rotations(circuit, n)
    
def swap_registers(circuit, n):
    """Swap qubits to reverse the order"""
    for qubit in range(n//2):
        circuit.swap(qubit, n-qubit-1)
    return circuit

def qft(circuit, n):
    """QFT on the first n qubits in circuit"""
    qft_rotations(circuit, n)
    swap_registers(circuit, n)
    return circuit

In [ ]:
# Create a QFT circuit for 3 qubits
n_qubits = 3
qc_qft = QuantumCircuit(n_qubits)

# Prepare an initial state (e.g., |1>)
qc_qft.x(0)

# Apply QFT
qft(qc_qft, n_qubits)

# Draw the circuit
print("\nQFT Circuit:")
print(qc_qft.draw())

In [ ]:
# Simulate and visualize
qc_qft_measure = qc_qft.copy()
qc_qft_measure.measure_all()

job = simulator.run(transpile(qc_qft_measure, simulator), shots=1024)
result = job.result()
counts = result.get_counts()

print("\nQFT Measurement Results:")
plot_histogram(counts)

---
# Lab 2: Bernstein-Vazirani Algorithm (BV)

The Bernstein-Vazirani algorithm finds a hidden binary string in a single query.

## BV Algorithm with Secret Number Input

Enter a secret binary string, and the algorithm will find it.

In [ ]:
def bv_oracle(circuit, secret_string):
    """Create the oracle for Bernstein-Vazirani algorithm"""
    n = len(secret_string)
    for i, bit in enumerate(reversed(secret_string)):
        if bit == '1':
            circuit.cx(i, n)
    return circuit

def bernstein_vazirani(secret_string):
    """Implement the Bernstein-Vazirani algorithm"""
    n = len(secret_string)
    
    # Create quantum circuit
    qc = QuantumCircuit(n+1, n)
    
    # Initialize auxiliary qubit in |-> state
    qc.x(n)
    qc.h(n)
    
    # Apply Hadamard gates to all input qubits
    for i in range(n):
        qc.h(i)
    
    qc.barrier()
    
    # Apply the oracle
    bv_oracle(qc, secret_string)
    
    qc.barrier()
    
    # Apply Hadamard gates to all input qubits
    for i in range(n):
        qc.h(i)
    
    # Measure
    for i in range(n):
        qc.measure(i, i)
    
    return qc

In [ ]:
# User input for secret number
# Example: secret_string = '10110101'
secret_string = '10110101'  # You can change this to any binary string

print(f"Secret string: {secret_string}")
print(f"Number of qubits: {len(secret_string)}")

# Create and display the circuit
qc_bv = bernstein_vazirani(secret_string)
print("\nBernstein-Vazirani Circuit:")
print(qc_bv.draw())

In [ ]:
# Simulate the circuit
job = simulator.run(transpile(qc_bv, simulator), shots=1024)
result = job.result()
counts = result.get_counts()

print(f"\nMeasurement results (should reveal the secret string: {secret_string}):")
plot_histogram(counts)

---
# Lab 3: Quantum Phase Estimation (QPE)

Quantum Phase Estimation is used to estimate the phase (eigenvalue) of an eigenvector of a unitary operator.

## QPE Implementation

We'll implement QPE for a simple T gate as an example.

In [ ]:
def qpe_pre(circuit, n_counting, n_eigen):
    """Prepare counting qubits in superposition and eigenstate"""
    # Initialize counting qubits in |+> state
    for qubit in range(n_counting):
        circuit.h(qubit)
    
    # Prepare eigenstate |1>
    circuit.x(n_counting)
    
    return circuit

def controlled_unitary_power(circuit, control_qubit, target_qubit, power):
    """Apply controlled-U^(2^power) where U is the T gate"""
    # For T gate: T = exp(i*pi/4*Z)
    # T^(2^power) = exp(i*pi/4*2^power*Z)
    angle = pi / 4 * (2 ** power)
    circuit.cp(angle, control_qubit, target_qubit)
    
def qpe_algorithm(n_counting):
    """Implement Quantum Phase Estimation"""
    n_eigen = 1  # One eigenstate qubit
    
    qc = QuantumCircuit(n_counting + n_eigen, n_counting)
    
    # Prepare the initial state
    qpe_pre(qc, n_counting, n_eigen)
    
    qc.barrier()
    
    # Apply controlled-U^(2^j) operations
    for j in range(n_counting):
        controlled_unitary_power(qc, j, n_counting, j)
    
    qc.barrier()
    
    # Apply inverse QFT
    qc_temp = QuantumCircuit(n_counting)
    qft(qc_temp, n_counting)
    qc.compose(qc_temp.inverse(), qubits=range(n_counting), inplace=True)
    
    # Measure counting qubits
    for i in range(n_counting):
        qc.measure(i, i)
    
    return qc

In [ ]:
# Create QPE circuit with 4 counting qubits
n_counting = 4
qc_qpe = qpe_algorithm(n_counting)

print("\nQuantum Phase Estimation Circuit:")
print(qc_qpe.draw())

In [ ]:
# Simulate QPE
job = simulator.run(transpile(qc_qpe, simulator), shots=2048)
result = job.result()
counts = result.get_counts()

print("\nQPE Measurement Results:")
print("Expected phase: 1/8 (for T gate eigenvalue)")
plot_histogram(counts)

In [ ]:
# Interpret the results
print("\nPhase Interpretation:")
for measured_state, count in sorted(counts.items(), key=lambda x: x[1], reverse=True)[:3]:
    decimal = int(measured_state, 2)
    phase = decimal / (2 ** n_counting)
    print(f"Measured: {measured_state} (decimal: {decimal}) -> Phase: {phase:.4f} (Count: {count})")

---
# Lab 4: Grover's Search on 3 Qubits

Grover's algorithm provides a quadratic speedup for unstructured search problems.

## Grover's Algorithm Implementation

We'll search for a specific state in a 3-qubit system.

In [ ]:
def initialize_s(qc, qubits):
    """Apply Hadamard gates to put qubits in superposition"""
    for q in qubits:
        qc.h(q)
    return qc

def oracle_grover(qc, marked_state):
    """Oracle that marks the target state with a phase flip"""
    # Convert marked_state to binary
    n = qc.num_qubits
    binary_state = format(marked_state, f'0{n}b')
    
    # Flip qubits that should be 0 in the target state
    for qubit, bit in enumerate(reversed(binary_state)):
        if bit == '0':
            qc.x(qubit)
    
    # Multi-controlled Z gate
    qc.h(n-1)
    qc.mcx(list(range(n-1)), n-1)
    qc.h(n-1)
    
    # Flip qubits back
    for qubit, bit in enumerate(reversed(binary_state)):
        if bit == '0':
            qc.x(qubit)
    
    return qc

def diffusion_operator(qc, qubits):
    """Apply the diffusion operator (inversion about average)"""
    n = len(qubits)
    
    # Apply H gates
    for q in qubits:
        qc.h(q)
    
    # Apply X gates
    for q in qubits:
        qc.x(q)
    
    # Multi-controlled Z
    qc.h(qubits[-1])
    qc.mcx(qubits[:-1], qubits[-1])
    qc.h(qubits[-1])
    
    # Apply X gates
    for q in qubits:
        qc.x(q)
    
    # Apply H gates
    for q in qubits:
        qc.h(q)
    
    return qc

def grover_algorithm(n_qubits, marked_state, n_iterations=None):
    """Implement Grover's algorithm"""
    if n_iterations is None:
        # Optimal number of iterations
        n_iterations = int(np.pi / 4 * np.sqrt(2 ** n_qubits))
    
    qc = QuantumCircuit(n_qubits, n_qubits)
    qubits = list(range(n_qubits))
    
    # Initialize superposition
    initialize_s(qc, qubits)
    
    qc.barrier()
    
    # Apply Grover iterations
    for _ in range(n_iterations):
        # Apply oracle
        oracle_grover(qc, marked_state)
        qc.barrier()
        
        # Apply diffusion
        diffusion_operator(qc, qubits)
        qc.barrier()
    
    # Measure
    qc.measure(qubits, qubits)
    
    return qc

In [ ]:
# Grover's search for state |101> (decimal 5) on 3 qubits
n_qubits = 3
marked_state = 5  # Binary: 101

qc_grover = grover_algorithm(n_qubits, marked_state)

print(f"\nSearching for state: {marked_state} (binary: {format(marked_state, f'0{n_qubits}b')})")
print("\nGrover's Algorithm Circuit:")
print(qc_grover.draw())

In [ ]:
# Simulate Grover's algorithm
job = simulator.run(transpile(qc_grover, simulator), shots=1024)
result = job.result()
counts = result.get_counts()

print("\nGrover's Search Results:")
print(f"Target state: {format(marked_state, f'0{n_qubits}b')}")
plot_histogram(counts)

---
# Lab 5: Shor's Factoring Algorithm

Shor's algorithm can factor large numbers exponentially faster than classical algorithms.

## Classical Helper Functions for Shor's Algorithm

In [ ]:
def find_period_classical(a, N):
    """Find the period r where a^r mod N = 1 (classical method for verification)"""
    result = 1
    for r in range(1, N):
        result = (result * a) % N
        if result == 1:
            return r
    return None

def shors_classical_postprocessing(N, a, measured_phase):
    """Process the quantum result to find factors"""
    # Convert phase to fraction
    frac = Fraction(measured_phase).limit_denominator(N)
    r = frac.denominator
    
    print(f"\nPhase = {measured_phase}")
    print(f"Fraction = {frac}")
    print(f"Candidate period r = {r}")
    
    # Check if r is even and a^(r/2) ≠ -1 mod N
    if r % 2 == 0:
        guesses = [gcd(a**(r//2) - 1, N), gcd(a**(r//2) + 1, N)]
        
        for guess in guesses:
            if guess not in [1, N] and N % guess == 0:
                return guess, N // guess
    
    return None, None

## Quantum Period Finding (Core of Shor's Algorithm)

In [ ]:
def c_amod15(a, power):
    """Controlled multiplication by a mod 15"""
    if a not in [2, 4, 7, 8, 11, 13]:
        raise ValueError("'a' must be coprime to 15")
    
    U = QuantumCircuit(4)
    
    for _ in range(power):
        if a in [2, 13]:
            U.swap(0, 1)
            U.swap(1, 2)
            U.swap(2, 3)
        if a in [7, 8]:
            U.swap(2, 3)
            U.swap(1, 2)
            U.swap(0, 1)
        if a in [4, 11]:
            U.swap(1, 3)
            U.swap(0, 2)
        if a in [7, 11, 13]:
            for q in range(4):
                U.x(q)
    
    U = U.to_gate()
    U.name = f"{a}^{power} mod 15"
    c_U = U.control()
    return c_U

def qpe_amod15(a, n_counting=8):
    """Quantum Phase Estimation for period finding in Shor's algorithm"""
    n_working = 4  # Working qubits for mod 15
    
    qc = QuantumCircuit(n_counting + n_working, n_counting)
    
    # Initialize counting qubits in |+> state
    for q in range(n_counting):
        qc.h(q)
    
    # Initialize working register to |1>
    qc.x(n_counting)
    
    # Apply controlled-U operations
    for q in range(n_counting):
        qc.append(c_amod15(a, 2**q), [q] + list(range(n_counting, n_counting + n_working)))
    
    qc.barrier()
    
    # Apply inverse QFT
    qc_qft = QuantumCircuit(n_counting)
    qft(qc_qft, n_counting)
    qc.compose(qc_qft.inverse(), qubits=range(n_counting), inplace=True)
    
    # Measure
    qc.measure(range(n_counting), range(n_counting))
    
    return qc

## Complete Shor's Algorithm to Factor N=15

In [ ]:
# Factor N = 15 using Shor's algorithm
N = 15
a = 7  # Choose a coprime to N (gcd(a, N) = 1)

print(f"Factoring N = {N} using a = {a}")
print(f"gcd({a}, {N}) = {gcd(a, N)}")

# Verify the classical period
r_classical = find_period_classical(a, N)
print(f"\nClassical period r = {r_classical}")
print(f"Verification: {a}^{r_classical} mod {N} = {pow(a, r_classical, N)}")

In [ ]:
# Create quantum circuit for period finding
n_counting = 8
qc_shor = qpe_amod15(a, n_counting)

print("\nShor's Algorithm Circuit (Period Finding):")
print(f"Circuit depth: {qc_shor.depth()}")
print(f"Number of qubits: {qc_shor.num_qubits}")

In [ ]:
# Simulate the circuit
job = simulator.run(transpile(qc_shor, simulator), shots=2048)
result = job.result()
counts = result.get_counts()

print("\nShor's Algorithm Results:")
plot_histogram(counts)

In [ ]:
# Process results to find factors
print("\nProcessing quantum measurement results...\n")

measured_phases = []
for measured_state, count in sorted(counts.items(), key=lambda x: x[1], reverse=True)[:5]:
    decimal = int(measured_state, 2)
    phase = decimal / (2 ** n_counting)
    measured_phases.append((phase, count))
    print(f"Measured: {measured_state} -> Phase: {phase:.4f} (Count: {count})")

print("\n" + "="*50)
print("Factor candidates:")
print("="*50)

factors_found = set()
for phase, count in measured_phases:
    if phase != 0:
        factor1, factor2 = shors_classical_postprocessing(N, a, phase)
        if factor1 and factor2:
            factors_found.add(tuple(sorted([factor1, factor2])))
            print(f"\n✓ Found factors: {factor1} × {factor2} = {N}")
            print(f"  Verification: {factor1} × {factor2} = {factor1 * factor2}")

if not factors_found:
    print("\nNo valid factors found in this run. Try running again or use different 'a' value.")
else:
    print(f"\n\nSUCCESS! Factored {N} = {' = '.join([f'{f1} × {f2}' for f1, f2 in factors_found])}")

---
## Summary

This notebook demonstrated five fundamental quantum algorithms:

1. **Quantum Fourier Transform (QFT)**: The quantum analogue of the DFT, essential for many quantum algorithms
2. **Bernstein-Vazirani (BV)**: Finds a hidden binary string in a single query, demonstrating quantum query advantage
3. **Quantum Phase Estimation (QPE)**: Estimates eigenvalues of unitary operators, used in many quantum algorithms
4. **Grover's Search**: Provides quadratic speedup for unstructured search problems
5. **Shor's Factoring**: Factors integers exponentially faster than known classical algorithms

Each algorithm showcases different aspects of quantum computing's power and potential applications.